In [0]:
from pyspark.sql import SparkSession
from datetime import datetime

from quality.engine.rules import Rules
from quality.engine.engine import Engine

from file_manager.move import FileManager


spark = SparkSession.getActiveSession()


LANDING_PATH = "/Volumes/workspace/callcenter/disc_volumes/landing/"


data_hoje = datetime.now().strftime("%Y%m%d")


arquivos = dbutils.fs.ls(LANDING_PATH)


for arquivo in arquivos:

    if not arquivo.path.endswith(".csv"):
        continue

    if f"discagem_{data_hoje}_" not in arquivo.path:
        continue

    df = (
        spark.read
        .option("header", True)
        .option("sep", ";")
        .csv(arquivo.path)
    )

    resultados = [

        Rules.validar_telefone_nulo(df),
        Rules.validar_data(df)

    ]

    payload = Engine.processar_resultados(
        resultados=resultados,
        nm_arquivo=arquivo.path
    )

    if payload["status"] == "aprovado":

        FileManager.move_to_approved(
            source_path=arquivo.path
        )

    else:

        FileManager.move_to_rejected(
            source_path=arquivo.path
        )